In [ ]:
5+5

# Is current flight algoirhtm finding the truely last message in sequence

# How far is the true airport from the last point of the flight

In [22]:
from data_engineering.adsb.read_adsb import read_adsb
from data_engineering.flights.flight_type import get_flights
from data_engineering.flights.sfdps_to_flights import get_sfdps_flights_day
from datetime import date
import polars as pl
target_date = date(2026, 3, 1)
df_sfdps = get_sfdps_flights_day(target_date)
df_test = get_flights(target_date)
df_test = df_test.filter(pl.col("pia") | pl.col("ladd"))
df_test = df_test.filter(pl.col("icao").is_in(df_sfdps.select("icao").to_series().implode()))


In [4]:
from datetime import date
target_date = date(2026,3,1)
from evaluation.flights.evaluation import main
outputs = main("model", "sfdps", target_date, pia_or_american_ladd_only=True)
df, stats= outputs[0]

Stats for model:sfdps: {'true_positive': 813, 'false_positive': 9, 'false_negative': 11, 'precision': 0.9890510948905109, 'recall': 0.9866504854368932, 'f1': 0.9878493317132442, 'airport_ident_match_pct': 0.95079950799508}
Stats CSV saved to /Volumes/T2-SSD/planequery/data/experiments/comparison/2026-06-27_23-49_f67d6bc_flights_evaluation.csv


In [8]:
df.columns

['icao',
 'callsign',
 'registration',
 'takeoff_time',
 'takeoff_airport_ident',
 'landing_time',
 'landing_airport_ident',
 'first_message_time',
 'first_lat',
 'first_lon',
 'first_baro_altitude_ft',
 'first_geom_altitude_ft',
 'last_message_time',
 'last_lat',
 'last_lon',
 'last_baro_altitude_ft',
 'last_geom_altitude_ft',
 'pia',
 'ladd',
 'military',
 'interesting',
 'aircraft_type',
 'owner',
 'aircraft_description',
 'category',
 'icao_df1',
 'callsign_df1',
 'registration_df1',
 'takeoff_time_df1',
 'takeoff_airport_ident_df1',
 'landing_time_df1',
 'landing_airport_ident_df1',
 'pia_df1',
 'ladd_df1',
 'military_df1',
 'interesting_df1',
 'aircraft_type_df1',
 'owner_df1',
 'aircraft_description_df1',
 'category_df1',
 'first_message_time_df1',
 'first_lat_df1',
 'first_lon_df1',
 'last_message_time_df1',
 'last_lat_df1',
 'last_lon_df1',
 'discard_df1',
 'takeoff_airport_lat_df1',
 'takeoff_airport_lon_df1',
 'landing_airport_lat_df1',
 'landing_airport_lon_df1',
 'match_st

In [10]:
import polars as pl
df_1 = df.filter(~pl.col("same_airport_ident"))
display(df_1.get_column("first_message_distance_to_takeoff_airport_km").describe())
display(df_1.get_column("last_message_distance_to_landing_airport_km").describe())

statistic,value
str,f64
"""count""",40.0
"""null_count""",20.0
"""mean""",9.501307
"""std""",21.135381
"""min""",0.081966
"""25%""",0.764698
"""50%""",2.080485
"""75%""",7.931051
"""max""",106.988849


statistic,value
str,f64
"""count""",40.0
"""null_count""",20.0
"""mean""",31.835527
"""std""",38.904993
"""min""",0.455305
"""25%""",4.543279
"""50%""",18.190515
"""75%""",39.675549
"""max""",199.611002


In [14]:
COLUMNS = [
    "icao",
    "first_message_time",
    "last_message_time",
    "first_message_distance_to_takeoff_airport_km",
    "last_message_distance_to_landing_airport_km",
    "takeoff_time_df1",
    "landing_time_df1",
    "takeoff_airport_ident",
    "landing_airport_ident",
    "takeoff_airport_ident_df1",
    "landing_airport_ident_df1",
    "same_airport_ident"
]

In [18]:
df.select(COLUMNS).filter(~pl.col("same_airport_ident")).filter(pl.col("first_message_distance_to_takeoff_airport_km") > 20.0)

icao,first_message_time,last_message_time,first_message_distance_to_takeoff_airport_km,last_message_distance_to_landing_airport_km,takeoff_time_df1,landing_time_df1,takeoff_airport_ident,landing_airport_ident,takeoff_airport_ident_df1,landing_airport_ident_df1,same_airport_ident
str,datetime[ms],datetime[ms],f64,f64,datetime[ms],datetime[ms],str,str,str,str,bool
"""a1a7dd""",2026-03-01 15:55:59.820,2026-03-01 16:50:19.850,81.748122,1.452045,2026-03-01 15:21:00,2026-03-01 16:50:00,"""KEND""","""KPWA""","""KPWA""","""KPWA""",false
"""a232fd""",2026-03-01 19:56:13.520,2026-03-01 20:34:22.360,26.371775,1.531804,2026-03-01 19:47:00,2026-03-01 20:38:00,"""KPGD""","""KPIE""","""KFMY""","""KPIE""",false
"""a6ab89""",2026-03-01 18:31:48.180,2026-03-01 19:20:35.930,21.494605,1.776208,2026-03-01 18:28:00,2026-03-01 19:22:00,"""US-3409""","""KJVY""","""KCEU""","""KJVY""",false
"""ad39a3""",2026-03-01 20:24:58.650,2026-03-01 21:04:39.210,24.353243,3.3365,2026-03-01 20:19:00,2026-03-01 21:08:00,"""94FL""","""KTMB""","""KPGD""","""KTMB""",false
"""adab24""",2026-03-01 19:10:52.880,2026-03-01 20:00:42.840,106.988849,4.543279,2026-03-01 18:11:00,2026-03-01 20:02:00,"""KCGC""","""KPIE""","""KPIE""","""KPIE""",false


In [24]:
df.select(COLUMNS).filter(pl.col("icao") == "a6ab89")

icao,first_message_time,last_message_time,first_message_distance_to_takeoff_airport_km,last_message_distance_to_landing_airport_km,takeoff_time_df1,landing_time_df1,takeoff_airport_ident,landing_airport_ident,takeoff_airport_ident_df1,landing_airport_ident_df1,same_airport_ident
str,datetime[ms],datetime[ms],f64,f64,datetime[ms],datetime[ms],str,str,str,str,bool
"""a6ab89""",2026-03-01 01:46:58.050,2026-03-01 02:46:44.490,0.276194,1.486207,2026-03-01 01:53:00,2026-03-01 02:47:00,"""KSAT""","""KDAL""","""KSAT""","""KDAL""",true
"""a6ab89""",2026-03-01 14:43:13.170,2026-03-01 16:20:28.990,1.601017,6.655689,2026-03-01 14:45:00,2026-03-01 16:25:00,"""KDAL""","""SC24""","""KDAL""","""KCEU""",false
"""a6ab89""",2026-03-01 18:31:48.180,2026-03-01 19:20:35.930,21.494605,1.776208,2026-03-01 18:28:00,2026-03-01 19:22:00,"""US-3409""","""KJVY""","""KCEU""","""KJVY""",false
"""a6ab89""",2026-03-01 20:14:18.400,2026-03-01 21:46:47.700,0.525799,1.50275,2026-03-01 20:14:00,2026-03-01 21:49:00,"""KJVY""","""KSWF""","""KJVY""","""KSWF""",true


In [25]:
from evaluation.flights.visual import visualize_flight

icao = "a6ab89"
visualize_flight(df_test.filter(pl.col("icao") == icao), read_adsb(target_date, icaos=[icao]))